## Preprocessing

preprocessing.py

### This Preprocessing.py  handles:

* Automatic energy column detection

* Column standardization

* Missing value handling

* Outlier treatment

* Feature scaling

The preprocessing module standardizes column names, handles missing values using forward and backward filling, caps extreme outliers at the 99th percentile to reduce skewness, and applies Min-Max normalization to prepare the dataset for anomaly detection models.

## Import Libraries 

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Function

In [ ]:
detect_energy_columns(df)

* Automatically finds energy-related columns in any dataset.

* Instead of hardcoding "electricity" or "kwh", it intelligently searches column names.

How It Works: 

In [ ]:
possible_energy_keywords = [
    "electricity",
    "energy",
    "power",
    "consumption",
    "kwh"
]

It defines keywords that commonly appear in energy datasets.

Then:

In [ ]:
for col in df.columns:
    for keyword in possible_energy_keywords:
        if keyword.lower() in col.lower():
            energy_cols.append(col)

It checks:

If any keyword appears inside a column name (case insensitive)

Example:

If your dataset has:

In [ ]:
Building_Electricity_Consumption

It will detect it automatically.

* Output

In [ ]:
Returns:

["electricity"]

or

["building_energy_kwh"]



This makes your system dataset-independent and scalable.

Very smart design 

##  Function : standardize_columns(df)

In [ ]:
def standardize_columns(df):


Cleans and standardizes column names.

This ensures different datasets follow a consistent naming convention.

In [ ]:
df.columns = df.columns.str.strip().str.lower()

Convert to lowercase & remove spaces:

In [ ]:
if "temp" in col:
    rename_map[col] = "temperature"

Rename similar columns to standard names:

Why This Is Important

Different datasets use different naming styles.

This makes our pipeline:

* Cleaner

* More robust

* More automated

## Function : preprocess_data(df)

In [ ]:
def preprocess_data(df):

This is the main cleaning function.

#### Step 1: Convert Timestamp

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

Convert Timestamp

Ensures timestamp is proper datetime type.

Then:

In [ ]:
df = df.sort_values("timestamp")

Sorts chronologically.

Why?

Because:

* Rolling features need ordered time

* Lag features need ordered time

#### Step 2: Handle Missing Values

In [ ]:
df = df.ffill().bfill()

ffill() → forward fill

bfill() → backward fill

Example: If one value is missing:

10, 12, NaN, 15

Becomes:

10, 12, 12, 15

This avoids dropping data.

#### Step 3: Cap Outliers

In [ ]:
upper = df[col].quantile(0.99)
df[col] = np.clip(df[col], None, upper)

This:

* Finds 99th percentile

* Caps extreme values above that

Why?

* Extreme spikes can:

* Distort model training

* Break scaling

* Bias anomaly detection

This keeps distribution realistic.

#### Step 4: Scale Features (0–1 Normalization)

In [ ]:
scaler = MinMaxScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

MinMaxScaler formula:

𝑋(𝑠𝑐𝑎𝑙𝑒𝑑)  =  X - min / max - min
=

Now every numeric column is between:

Why this matters:

* Isolation Forest works better

* SVM requires scaling

* Distance-based models need normalized inputs

#### Step 5: Validation Checks

In [ ]:
Missing Values
Timestamp Sorted
Min value after scaling
Max value after scaling

This ensures preprocessing worked correctly.

## Return

In [ ]:
  return df

Return the df to Streamlit Dashboard

## Whole Code :

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler


def detect_energy_columns(df):

    possible_energy_keywords = [
        "electricity",
        "energy",
        "power",
        "consumption",
        "kwh"
    ]

    energy_cols = []

    for col in df.columns:
        for keyword in possible_energy_keywords:
            if keyword.lower() in col.lower():
                energy_cols.append(col)

    return energy_cols
def standardize_columns(df):

    df.columns = df.columns.str.strip().str.lower()

    rename_map = {}

    for col in df.columns:
        if "temp" in col:
            rename_map[col] = "temperature"
        if "humid" in col:
            rename_map[col] = "humidity"
        if "time" in col:
            rename_map[col] = "timestamp"

    df = df.rename(columns=rename_map)

    return df


def preprocess_data(df):

    print("🧹 Preprocessing...")

    # Convert timestamp
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp")

    # Fill missing
    df = df.ffill().bfill()

    # Cap outliers at 99 percentile
    numeric_cols = df.select_dtypes(include=np.number).columns

    for col in numeric_cols:
        upper = df[col].quantile(0.99)
        df[col] = np.clip(df[col], None, upper)

    # Scale to 0-1
    scaler = MinMaxScaler()
    df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

    print("✅ Preprocessing Completed")
    # Validation checks
    print("🔎 Missing Values:", df.isna().sum().sum())

    print("🔎 Timestamp Sorted:",
      df["timestamp"].is_monotonic_increasing)

    numeric_cols = df.select_dtypes(include='number').columns

    print("🔎 Min Value After Scaling:",
      df[numeric_cols].min().min())

    print("🔎 Max Value After Scaling:",
      df[numeric_cols].max().max())
    return df